In [5]:
import igraph as ig
import leidenalg as la
import cairo
import os
import palettable

net_graph_all_pos = ig.Graph()
output_dir = "/Users/htang/Documents/PhD/Project - CAS/C3_Microbiome/2019-10-24_QIIME_CAS_COAST_meta_5/" + \
    "output/2020-09-25_fastspar_community/"
net_graph_all_pos_gml = net_graph_all_pos.Read_GML(output_dir + "net.graph.all.pos.txt")
ig.summary(net_graph_all_pos_gml)

IGRAPH UNW- 532 23274 -- 
+ attr: id (v), name (v), weight (e)


In [29]:
net_graph_all_pos_layout = net_graph_all_pos_gml.layout_fruchterman_reingold()
color_tuple = tuple(palettable.colorbrewer.qualitative.Set3_12.colors[1])
color_tuple2 = tuple(x/255 for x in tuple(palettable.colorbrewer.qualitative.Set3_12.colors[1]))
color_tuple2

SyntaxError: Generator expression must be parenthesized (<ipython-input-29-215dfe6e9484>, line 3)

In [30]:
x = 1
res_para = 1
# for y in range(5):
#     # Resolution paramaters: 2, 1, 0.5, 0.25, 0.125
#     res_para = 2 ** (-y+1)
#     for x in range(100):
net_graph_all_pos_part = la.find_partition(net_graph_all_pos_gml, \
                                           la.RBConfigurationVertexPartition, \
                                           seed = x, resolution_parameter = res_para)
# Get number of partitions
net_graph_all_pos_part_num = len(set(net_graph_all_pos_part.membership))
color_dict = {cluster: tuple(x/255 for x in tuple(palettable.colorbrewer.qualitative.Set3_12.colors[cluster])) for \
               cluster in range(net_graph_all_pos_part_num)}
color_dict
# pal = ig.drawing.colors.RainbowPalette(n=net_graph_all_pos_part_num)
# color_dict = {cluster: pal.get(cluster) for cluster in range(net_graph_all_pos_part_num)}

# ig.plot(net_graph_all_pos_gml, layout=net_graph_all_pos_layout, \
#                                    palette=pal)
# type(pal.get(2))
# pal.get(2)

{0: (0.5529411764705883, 0.8274509803921568, 0.7803921568627451),
 1: (1.0, 1.0, 0.7019607843137254),
 2: (0.7450980392156863, 0.7294117647058823, 0.8549019607843137),
 3: (0.984313725490196, 0.5019607843137255, 0.4470588235294118)}

In [33]:
# x = 1
# res_para = 1
for y in range(5):
    # Resolution paramaters: 2, 1, 0.5, 0.25, 0.125
    res_para = 2 ** (-y+1)
    for x in range(100):
        net_graph_all_pos_part = la.find_partition(net_graph_all_pos_gml, \
                                                   la.RBConfigurationVertexPartition, \
                                                   seed = x, resolution_parameter = res_para)
        # Get number of partitions
        net_graph_all_pos_part_num = len(set(net_graph_all_pos_part.membership))
        # Get palette
        # color_dict = {0: 'grey', 1: 'red', 2: 'orange', 3: 'yellow', 4: 'green', 5: 'blue', 6: 'purple'}
        pal = ig.drawing.colors.RainbowPalette(n=net_graph_all_pos_part_num)
        color_dict = {cluster: pal.get(cluster) for cluster in range(net_graph_all_pos_part_num)}
        # color_dict = {cluster: tuple(x/255 for x in tuple(palettable.colorbrewer.qualitative.Set3_12.colors[cluster])) for \
        #              cluster in range(net_graph_all_pos_part_num)}
        net_graph_all_pos_gml.vs["label"] = net_graph_all_pos_part.membership
        net_graph_all_pos_gml.vs["color"] = [color_dict[cluster] for cluster in net_graph_all_pos_gml.vs["label"]]

        # net_graph_all_pos_gml.vs["color"] = [color_dict[module] for module in net_graph_all_pos_gml.vs["label"]]
        net_graph_all_pos_gml_plot = ig.plot(net_graph_all_pos_gml, layout=net_graph_all_pos_layout)
        # net_graph_all_pos_gml_plot.surface
        # Subdirectory for leiden output
        output_subdir = output_dir + "leiden/" + str(res_para) + "/"
        if not os.path.exists(output_subdir):
            os.makedirs(output_subdir)
        # Save plot
        net_graph_all_pos_gml_plot.save(output_subdir + "net.graph.all.pos.node.names." + str(x) + ".png")
        # Save partitions
        data = net_graph_all_pos_part
        with open(output_subdir + "net.graph.all.pos.node.parts." + str(x) + ".txt", "w") as txt_file:
            for line in data:
                txt_file.write("%s\n" % line)

In [ ]:
# Save namelist of nodes/vertices
data = net_graph_all_pos_gml.vs["name"]
with open(output_dir + "net.graph.all.pos.node.names.txt", "w") as txt_file:
    for line in data:
        txt_file.write("%s\n" % line)